# Agent Personnel (OpenAI Function Calling)

**But :** prototype d'un agent personnel qui utilise la Function Calling d'OpenAI, un vector store pour mémoire, et expose un endpoint simple.

**Prérequis :**
- Avoir une clé `OPENAI_API_KEY` définie dans l'environnement ou dans un fichier `.env`.
- Installer les packages listés ci‑dessous.

**Attention :** les appels réels à l'API OpenAI sont désactivés dans ce notebook de démonstration. Remplace les placeholders et installe la librairie `openai` pour exécuter. 

## Installation (terminal)
```bash
pip install openai fastapi uvicorn python-dotenv requests aiohttp nbformat
# optional for vector DB: pip install pinecone-client weaviate-client
```

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('OPENAI_API_KEY set:', bool(OPENAI_API_KEY))

## 1) Définir les functions (JSON Schema) que l'agent peut appeler
On transmettra ces schemas au modèle via la paramètre `functions` de l'API. Ils doivent décrire le nom, la description et les paramètres attendus.

In [ ]:
functions = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. Paris"},
                "country": {"type": "string", "description": "Country code, optional"}
            },
            "required": ["city"]
        }
    },
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with title, start and end times",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start_iso": {"type": "string"},
                "end_iso": {"type": "string"},
                "participants": {"type": "array", "items": {"type": "string"}}
            },
            "required": ["title", "start_iso", "end_iso"]
        }
    }
]

len(functions)

## 2) Client OpenAI - exemple d'appel avec function calling
La cellule suivante contient un wrapper. **Ne l'exécute que si** tu as installé `openai` et défini `OPENAI_API_KEY`.
En production, remplace `model` par le modèle souhaité et ajuste les paramètres (temperature, max_tokens...).

In [ ]:
try:
    import openai
except Exception as e:
    openai = None
    print('openai not installed; install with pip install openai to enable real API calls.')

import json, os
def call_openai_chat(messages, functions):
    if openai is None:
        # Mocked response structure
        return {'mock': True, 'choices': [{'message': {'role': 'assistant', 'content': None, 'function_call': {'name': 'get_weather', 'arguments': json.dumps({'city':'Paris'})}}}], 'usage': {}}
    openai.api_key = os.getenv('OPENAI_API_KEY')
    resp = openai.ChatCompletion.create(
        model='gpt-4o-mini',
        messages=messages,
        functions=functions,
        function_call='auto',
        temperature=0.2,
        max_tokens=800
    )
    return resp

# Test wrapper (mocked if openai not installed)
messages = [{"role":"user","content":"Quel temps fera-t-il à Paris demain ?"}]
resp = call_openai_chat(messages, functions)
print('Response preview:', json.dumps(resp if isinstance(resp, dict) else {'ok':True}, default=str)[:1000])

## 3) Tools (mock) - implementations locales
Ces fonctions simulent des outils. Remplace-les par de véritables intégrations (API météo, Google Calendar...) pour un usage réel.

In [ ]:
def get_weather_tool(city, country=None):
    return {"city": city, "country": country or "FR", "temp_c": 15, "condition": "Partly cloudy"}

def create_calendar_event_tool(title, start_iso, end_iso, participants=None):
    return {"event_id": "evt_12345", "title": title, "start": start_iso, "end": end_iso, "participants": participants or []}

def dispatch_function_call(name, arguments):
    # arguments may be a dict or a JSON string
    if isinstance(arguments, str):
        try:
            import json as _json
            arguments = _json.loads(arguments)
        except:
            arguments = {}
    if name == 'get_weather':
        return get_weather_tool(**arguments)
    if name == 'create_calendar_event':
        return create_calendar_event_tool(**arguments)
    return {'error':'unknown function'}

# Quick demo
print(dispatch_function_call('get_weather', {'city':'Paris'}))

## 4) Boucle d'interaction agent -> function -> retour au modèle
Logique :
1. Envoie du message utilisateur au modèle
2. Si modèle demande function_call, exécuter la fonction locale
3. Renvoyer la sortie de la fonction au modèle comme message de rôle `function`
4. Obtenir la réponse finale pour l'utilisateur

In [ ]:
# Simulated interaction loop
history = [
    {'role':'system','content':'Tu es un assistant qui peut appeler des fonctions pour aider l utilisateur.'},
    {'role':'user','content':'Programme un rendez-vous demain à 10h avec Alice.'}
]

# Model (mock) indicates function call
model_decision = {'name':'create_calendar_event','arguments':{'title':'Réunion suivi','start_iso':'2025-10-15T10:00:00','end_iso':'2025-10-15T11:00:00','participants':['alice@example.com']}}
tool_result = dispatch_function_call(model_decision['name'], model_decision['arguments'])
# Append function result as message
history.append({'role':'function','name':model_decision['name'],'content':json.dumps(tool_result)})
print('History after function result:')
print(history)

## 5) Exposer l'agent via FastAPI (squelette)
Ce serveur reçoit une requête utilisateur et renvoie la réponse de l'agent. À sécuriser et adapter en production.

In [ ]:
# FastAPI skeleton (run with: uvicorn Agent_Personnel:app --reload)
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import json

app = FastAPI()

class ChatRequest(BaseModel):
    user_input: str

@app.post('/chat')
async def chat(req: ChatRequest):
    try:
        messages = [{'role':'system','content':'Tu es un assistant.'}, {'role':'user','content':req.user_input}]
        resp = call_openai_chat(messages, functions)
        # If mocked, emulate function call handling
        if isinstance(resp, dict) and resp.get('mock'):
            choice = resp['choices'][0]['message']
            fc = choice.get('function_call')
            if fc:
                result = dispatch_function_call(fc['name'], fc.get('arguments', {}))
                return {'result': result}
        return {'response': 'Réponse du modèle (ou mock)'}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Note: pour exécuter localement, sauvegarde ce fichier et lance uvicorn.")

## 6) Métriques simples et monitoring
- Mesurer tokens utilisés, latence, erreurs
- Loguer les appels fonction et résultats
- Calculer coût estimé par requête (tokens * unit price)

In [ ]:
# Simple logging example
import time

def simple_agent_call(user_input):
    start = time.time()
    resp = call_openai_chat([{'role':'user','content':user_input}], functions)
    duration = time.time() - start
    # Mock usage
    usage = {'prompt_tokens':50, 'completion_tokens':100}
    print(f'call duration: {duration:.2f}s, usage: {usage}')
    return resp

print('Sample call:')
print(simple_agent_call('Dis moi la meteo a Paris demain'))

## 7) Dockerfile et déploiement (exemple)
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY . /app
RUN pip install --no-cache-dir openai fastapi uvicorn python-dotenv
CMD ["uvicorn", "Agent_Personnel:app", "--host", "0.0.0.0", "--port", "8080"]
```


## 8) Checklist pour production
- Gestion des clés / vault
- Quotas et limites
- Sanitization des inputs
- Rejets (policies) et modération
- Tests adversariaux
- Monitoring et alerting

---

Fin du notebook 1.